# HW5. Короткий запуск на GPU

Выполняй на одном CUDA GPU в Docker-окружении из этого проекта. Все методы импортируются из .py. Этот notebook проверяет настоящую генерацию, callback награды, несколько шагов GRPO, изменение LoRA-весов, сохранение адаптера и расход памяти. Он не запускает BFCL и не является одним из трёх основных экспериментов.

Нужны доступ к Hugging Face и свободная GPU-память. Образ и SSH-туннель описаны в README.md. Данные и веса загружаются здесь на сервере, а не на локальной машине.

In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import json
import subprocess
from datetime import datetime
from pathlib import Path
from pprint import pprint

import torch

from download_data import download_data
from prepare import prepare_data
from train import run_training

subprocess.run(['nvidia-smi'], check=True)
print('PyTorch:', torch.__version__)
print('CUDA доступна:', torch.cuda.is_available())

/home/edu/amsafin/HSE-Avito/LLM/hw5/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sat Nov  7 23:31:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.86.15              Driver Version: 570.86.15      CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:06:00.0 Off |                    0 |
| N/A   32C    P0             62W /  400W |       1MiB /  81920MiB |      2%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Небольшой настоящий ToolACE

Для короткого запуска достаточно первых 200 диалогов. Разделение выполняется по диалогам до нарезки. В реальном обучении используются только ToolACE и сгенерированные текущей моделью ответы: подставленные локальные ответы сюда не передаются.

In [3]:
input_path = Path('data/raw/data.json')
if not input_path.exists():
    download_data()
report = prepare_data(input_path, 'data/prepared', limit=200)
print('Train-примеров:', report['train_examples'])
print('Validation-примеров:', report['validation_examples'])
print('Исключённых диалогов:', len(report['excluded_dialogues']))
pprint(report['excluded_dialogues'][:5])

Train-примеров: 843
Validation-примеров: 98
Исключённых диалогов: 5
[{'dialogue_id': 27, 'reason': 'Пустая реплика assistant: 1'},
 {'dialogue_id': 30, 'reason': 'Пустая реплика assistant: 1'},
 {'dialogue_id': 162, 'reason': 'Пустая реплика assistant: 1'},
 {'dialogue_id': 163, 'reason': 'Пустая реплика assistant: 1'},
 {'dialogue_id': 175, 'reason': 'invalid syntax (<unknown>, line 1)'}]


## 2. Конфигурация проверки

configs/smoke.json задаёт пять optimizer-шагов, четыре генерации на запрос, максимум 512 новых токенов и LoRA rank 8 на q_proj/v_proj. GRPO использует beta=0, чтобы не загружать вторую модель. Генерация идёт через Transformers, без отдельного экземпляра vLLM. Микробатч 1 и накопление 4 дают полный набор из четырёх ответов на запрос.

Награда: exact + 0.2 × functions + 0.05 × format. Это стартовая смесь для проверки механизма, а не выбранная лучшая конфигурация. Ответы, достигшие лимита без EOS, получают ноль и исключаются из loss. Prompt длиннее 2048 токенов исключается целиком с подсчётом; он не обрезается скрыто.

Каждая попытка использует отдельную папку. Уже записанные результаты train.py не перезаписывает.

In [4]:
config_path = Path('configs/smoke.json')
settings = json.loads(config_path.read_text())
pprint(settings)
output_dir = Path('results') / f"smoke_{datetime.now():%Y%m%d_%H%M%S}"
print('Результаты:', output_dir)

{'beta': 0.0,
 'dataset_limit': 16,
 'gradient_accumulation_steps': 4,
 'learning_rate': 2e-05,
 'lora_alpha': 16,
 'lora_rank': 8,
 'loss_type': 'grpo',
 'max_completion_length': 512,
 'max_prompt_tokens': 2048,
 'max_steps': 5,
 'num_generations': 4,
 'per_device_train_batch_size': 1,
 'reward_weights': {'exact': 1.0, 'format': 0.05, 'functions': 0.2},
 'scale_rewards': 'group',
 'seed': 42,
 'target_modules': ['q_proj', 'v_proj'],
 'temperature': 1.0}
Результаты: results/smoke_20261107_233150


## 3. Настоящее обучение

Здесь впервые загружаются веса исходной Qwen закреплённой ревизии. Обучаются только адаптеры. run_training сохраняет конфигурацию, реальные генерации с компонентами награды, историю метрик, адаптер и summary.json. Расход памяти измеряется средствами PyTorch на фактическом запуске.

In [5]:
summary = run_training(
    data_path='data/prepared/train.jsonl',
    config_path=config_path,
    output_dir=output_dir,
)
pprint(summary)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 320/320 [00:00<00:00, 7800.91it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 417,792 || all params: 1,882,242,880 || trainable%: 0.0222


Step,Training Loss
1,-0.374940
2,0.000000
3,-0.432953
4,0.000000
5,0.000000


{'changed_parameter_tensors': 24,
 'finite_training_metrics': True,
 'generated_completions': 20,
 'gpu': 'NVIDIA A100-SXM4-80GB',
 'gpu_free_before_load_gib': 78.838623046875,
 'gpu_total_gib': 79.251708984375,
 'groups_with_reward_variation': 3,
 'optimizer_steps': 5,
 'parameter_delta_l2': 0.027177104054622557,
 'peak_allocated_gib': 4.1297736167907715,
 'peak_reserved_gib': 4.533203125,
 'reward_groups': 5,
 'status': 'updated',
 'train_seconds': 189.81826742296107,
 'trainable_parameters': 417792,
 'truncated_completions': 3}


## 4. Посмотреть реальную генерацию и награды

generations.jsonl содержит именно генерации модели. Для каждой сохранены шаг, группа ответов на один запрос, эталон, компоненты награды, итоговая награда, длина, признак обрезания и выделенная GPU-память. history.json содержит метрики optimizer-шагов.

In [6]:
generations = []
for line in (output_dir / 'generations.jsonl').read_text().splitlines():
    generations.append(json.loads(line))
for generation in generations[:4]:
    print('Шаг:', generation['step'], 'группа:', generation['group'])
    print(generation['response'])
    print('Компоненты:', generation['scores'], 'награда:', generation['reward'])
    print('Обрезан:', generation['truncated'])
print('История обучения:')
pprint(json.loads((output_dir / 'history.json').read_text()))

Шаг: 0 группа: 0
Yes, managing travel expenses when gas prices are a major concern is a key part of budgeting for road trips this summer! Here are a few solid strategies to consider:

*   **Prioritize Fuel Use:** Plan your start and end points to minimize the time driving. If a destination requires a 2-hour layover at a stopover city (which is common for gas-rich stops like Bad давайте just suggested previous fuel stops), keep it in sight, or take a detour away from it.
*   **Localize Your Stops:** While gas stops not found in previous searches are recommended, minimize unnecessary kilometres by swapping stops with nearby locations that have those cheap fuel stations nearby (e.g., if a specific retail station isn't in your area, pick a station within 10km that has enough capacity for your expected fuel needs).
*   **Travel in Groups:** Sharing a car or a vehicle load can reduce per-person gas costs.
*   **Select The New Year One Destination:** If your plan involves visiting "TCL" or ev

## 5. Что считать успешной проверкой

Должны завершиться все optimizer-шаги; метрики должны быть конечными; реальные LoRA-параметры должны измениться. parameter_delta_l2 измеряет евклидову норму изменения обучаемых весов. peak_allocated_gib и peak_reserved_gib — максимальная выделенная и зарезервированная PyTorch память в GiB (байты / 1024³), включая загрузку модели.

Если все ответы в каждой группе получили одинаковую награду, advantage равен нулю и обучение может не изменить веса. В этом случае notebook не объявит успех: посмотри generations.jsonl, число обрезанных ответов и groups_with_reward_variation. Для следующей попытки увеличь число шагов или генераций, уточни награду либо лимит ответа по реальным выводам и создай новую папку результатов. Завершение процесса само по себе не доказывает обновление модели.

Сохранён adapter, а не полная объединённая модель. Для будущего BFCL нужен отдельный экспорт через train.export_model. Baseline и полноценные эксперименты находятся в experiments.ipynb.

In [7]:
print('Пик выделенной памяти, GiB:', summary['peak_allocated_gib'])
print('Пик зарезервированной памяти, GiB:', summary['peak_reserved_gib'])
print('Изменение обучаемых весов, L2:', summary['parameter_delta_l2'])
print('Групп с различными наградами:', summary['groups_with_reward_variation'])
assert summary['optimizer_steps'] == settings['max_steps']
assert summary['finite_training_metrics']
assert summary['generated_completions'] > 0
assert summary['peak_allocated_gib'] > 0
assert summary['changed_parameter_tensors'] > 0, 'Веса не изменились; проверь реальные награды'
print('Короткий запуск прошёл: генерация, награда, optimizer, адаптер и GPU-память проверены.')

Пик выделенной памяти, GiB: 4.1297736167907715
Пик зарезервированной памяти, GiB: 4.533203125
Изменение обучаемых весов, L2: 0.027177104054622557
Групп с различными наградами: 3
Короткий запуск прошёл: генерация, награда, optimizer, адаптер и GPU-память проверены.
